In [ ]:
import warnings
import numpy as np

import prospect.sources.nebssp_basis as nb
from prospect.sources import SSPBasis

from hubersed.fitting.chi2 import load_by_index, tids_to_indices, WAVE_OBS
from hubersed.fitting.config import build_continuum_model, build_full_cue_model
from hubersed.prospector.parameter_file import build_obs, build_cue_sps, build_sps
from hubersed.prospector.lsf import desi_resolution, C_KMS
from hubersed.conversion import flambda_to_maggies, ivar_flambda_to_ivar_maggies

In [ ]:
TID   = 39632991244258619          # "94183", the EELG
LSF   = (C_KMS / (2.355 * desi_resolution(WAVE_OBS))).astype(np.float64)
LINES = {"Hb": 4861.0, "[OIII]": 5007.0, "Ha": 6563.0}

_ORIG_SR = SSPBasis.__dict__.get("spectral_resolution")

def zero_library():
    SSPBasis.spectral_resolution = property(lambda self: np.zeros_like(self.ssp.wavelengths))

def restore_library():
    if _ORIG_SR is not None:
        SSPBasis.spectral_resolution = _ORIG_SR
    elif "spectral_resolution" in SSPBasis.__dict__:
        delattr(SSPBasis, "spectral_resolution")

In [ ]:
idx = int(tids_to_indices(np.array([TID], np.int64))[0])
spec, ivar, z, tid = load_by_index(idx)
assert int(tid) == TID, f"Loaded TID {tid} does not match expected {TID}."

In [ ]:
fm  = flambda_to_maggies(WAVE_OBS, spec)
iv  = ivar_flambda_to_ivar_maggies(WAVE_OBS, ivar)
ok  = (iv > 0) & np.isfinite(fm)
iv  = np.where(ok, iv, 0.0)
sig = 1.0 / np.sqrt(np.where(iv > 0, iv, np.inf))

In [ ]:
zero_library()
# sps = build_sps()
cue_sps = build_cue_sps()
assert SSPBasis.__dict__["spectral_resolution"] is not _ORIG_SR

obs = build_obs(spec=fm, unc=sig, mask=ok, resolution=LSF, wavelength=WAVE_OBS)
cont_model, cont_tmpl = build_continuum_model(z)
cue_model, cue_tmpl = build_full_cue_model(cont_tmpl, cont_model.theta, cont_model, z)

print(f"TID {tid}  z={z:.4f}  ndim={len(cue_model.theta)}")
print("free params:", cue_model.theta_labels())
print("nebemlineinspec :", cue_tmpl["nebemlineinspec"]["init"])
print("use_stellar_ionizing:", cue_tmpl["use_stellar_ionizing"]["init"])

In [ ]:
def predict_at(model, theta, sps=cue_sps, observations=obs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        preds, _ = model.predict(np.asarray(theta, float), observations=observations, sps=sps)
    return np.asarray(preds[0], float)

def compare(a, b, label, mask=ok):
    d = np.abs(a - b)[mask]
    scale = np.maximum(np.abs(a)[mask], 1e-300)
    out = {"label": label, "max_abs": float(np.nanmax(d)),
           "max_frac": float(np.nanmax(d / scale))}
    for name, lam in LINES.items():
        j = int(np.argmin(np.abs(WAVE_OBS - lam * (1 + z))))
        out[name] = float(abs(a[j]-b[j]) / a[j]) if a[j] != 0 else np.nan
    print(f"{label:<38} max|d|={out['max_abs']:.6e}  max frac={out['max_frac']:.6e}  "
          + "  ".join(f"{k}={out[k]:.4e}" for k in LINES))
    return out

In [ ]:
RESULTS = {}
th0 = cue_model.theta.copy()
i_qion = cue_model.theta_index["gas_logqion"]
print("gas_logqion init:", float(np.atleast_1d(th0[i_qion])[0]))

In [ ]:
p0a = predict_at(cue_model, th0)
p0b = predict_at(cue_model, th0)
RESULTS["0_determinism"] = compare(p0a, p0b, "TEST 0  same theta twice")
assert RESULTS["0_determinism"]["max_abs"] == 0.0, "predict is not deterministic; stop here"

In [ ]:
thA1, thA2 = th0.copy(), th0.copy()
thA1[i_qion] = 49.0
thA2[i_qion] = 52.0

pA1 = predict_at(cue_model, thA1)
pA2 = predict_at(cue_model, thA2)
RESULTS["A_stock"] = compare(pA1, pA2, "TEST A  gas_logqion 49 vs 52 (stock)")

print("\n  -> DEAD, claim confirmed." if RESULTS["A_stock"]["max_abs"] == 0.0
      else "\n  -> spectrum moved. Claim REFUTED; tell me.")

In [ ]:
_ORIG_FIT = nb.fit_4loglinear_ionparam
CALLS = []
_BOUNDS = {"ionspec_index1": (1.0, 42.0), "ionspec_index2": (-0.3, 30.0),
           "ionspec_index3": (-1.0, 14.0), "ionspec_index4": (-1.7, 8.0),
           "ionspec_logLratio1": (-1.0, 10.1), "ionspec_logLratio2": (-0.5, 1.9),
           "ionspec_logLratio3": (-0.4, 2.2)}

def _fit_recording(wav, spec, **kw):
    d = _ORIG_FIT(wav, spec, **kw)
    coeff = np.asarray(d["powerlaw_params"], float)     # (4,2): [slope, norm] per segment
    CALLS.append({"returned": {k: v for k, v in d.items() if k != "powerlaw_params"},
                  "raw_slopes": coeff[:, 0].copy()})
    return d

nb.fit_4loglinear_ionparam = _fit_recording
CALLS.clear()
_ = predict_at(cue_model, thA1)
nb.fit_4loglinear_ionparam = _ORIG_FIT

print(f"called {len(CALLS)}x per predict (young, old CSP)")
print("keys returned:", sorted(CALLS[0]["returned"].keys()))
print("'gas_logqion' in returned dict:", "gas_logqion" in CALLS[0]["returned"], "\n")
for n, c in enumerate(CALLS):
    tag = ["young", "old"][n] if n < 2 else str(n)
    print(f"  [{tag}] gas_logqion set to {c['returned']['gas_logqion']:.3f} "
          f"(your theta was {float(np.atleast_1d(thA1[i_qion])[0]):.3f})")
    for k in ("ionspec_index1","ionspec_index2","ionspec_index3","ionspec_index4"):
        lo, hi = _BOUNDS[k]
        raw  = float(c["raw_slopes"][int(k[-1]) - 1])
        post = float(np.atleast_1d(c["returned"][k])[0])
        flag = "  <-- CLIPPED" if not (lo <= raw <= hi) else ""
        print(f"        {k:<20} raw={raw:+10.3f}  used={post:+10.3f}  [{lo}, {hi}]{flag}")
    for k in ("ionspec_logLratio1","ionspec_logLratio2","ionspec_logLratio3"):
        lo, hi = _BOUNDS[k]
        post = float(np.atleast_1d(c["returned"][k])[0])
        edge = "  <-- AT BOUND" if np.isclose(post, lo) or np.isclose(post, hi) else ""
        print(f"        {k:<20} used={post:+10.3f}  [{lo}, {hi}]{edge}")

In [ ]:
def _fit_keep_shape_only(wav, spec, **kw):
    d = _ORIG_FIT(wav, spec, **kw)
    d.pop("gas_logqion", None)     # keep the shape, free theta owns the normalization
    return d

nb.fit_4loglinear_ionparam = _fit_keep_shape_only
pC1 = predict_at(cue_model, thA1)
pC2 = predict_at(cue_model, thA2)
RESULTS["C_patched"] = compare(pC1, pC2, "TEST C  gas_logqion 49 vs 52 (patched)")
nb.fit_4loglinear_ionparam = _ORIG_FIT     # restore

if RESULTS["C_patched"]["max_abs"] > 0 and RESULTS["A_stock"]["max_abs"] == 0.0:
    print("\n  -> machinery CAN see a qion change; Test A's zero is a real no-op.")
elif RESULTS["C_patched"]["max_abs"] == 0.0:
    print("\n  -> patched run also flat. Something else blocks it; distrust Test A too.")

In [ ]:
print("model params nebemlineinspec:", cue_model.params.get("nebemlineinspec"))
print("model        _need_lines    :", cue_model._need_lines, "\n")

for name, lo, hi in [("eline_sigma", 30.0, 240.0), ("sigma_smooth", 30.0, 300.0)]:
    if name not in cue_model.theta_index:
        print(f"{name}: not free, skipped"); continue
    j = cue_model.theta_index[name]
    t1, t2 = th0.copy(), th0.copy()
    t1[j], t2[j] = lo, hi
    RESULTS[f"D_{name}"] = compare(predict_at(cue_model, t1), predict_at(cue_model, t2),
                                   f"TEST D  {name} {lo:g} vs {hi:g}")

if RESULTS.get("D_eline_sigma", {}).get("max_abs") == 0.0 \
   and RESULTS.get("D_sigma_smooth", {}).get("max_abs", 0) > 0:
    print("\n  -> eline_sigma DEAD; sigma_smooth drives line widths. Cue lines broadened to")
    print("     the STELLAR dispersion. Set nebemlineinspec=False before comparing run1/run2.")

In [ ]:
print(f"{'test':<40} {'max|d|':>14} {'max frac':>14}")
for v in RESULTS.values():
    print(f"{v['label']:<40} {v['max_abs']:>14.6e} {v['max_frac']:>14.6e}")

restore_library()
print("\ngas_logqion is a NO-OP under cue_stellar_nebular:",
      RESULTS["A_stock"]["max_abs"] == 0.0 and RESULTS["C_patched"]["max_abs"] > 0.0)